# Notebook 04a_extended: Static CMF Unlearning — Extended (MAX_EPOCHS=50, PATIENCE=5) (`04a_extended_cmf_static.ipynb`)

### Objectives & Workflow:
1. **CMF Representation Unlearning Framework**: Apply the CMF mechanism across all 5 machine unlearning baselines:
   - **Random Label (RL)**
   - **NegGrad+**
   - **SalUn**
   - **SCRUB**
   - **TarUn / UNSIR**
2. **CMF Head Reconstruction & Freezing**: Reconstruct classifier weights directly from centered & normalized class-mean features of $D = D_R \cup D_F$ (zero bias) $\mathbf{w}_c = \frac{\bar{\mathbf{z}}_c - \bar{\mathbf{z}}}{\|\bar{\mathbf{z}}_c - \bar{\mathbf{z}}\|_2}$, freeze the linear head, and optimize only the encoder $\theta$ using each unlearning algorithm.
3. **Per-Epoch Re-Tying (Algorithm 2) & Early Stopping**: Rebuild the CMF head from $D = D_R \cup D_F$ after every unlearning epoch. Track Oracle proximity and validation criteria with early stopping across maximum epochs (**30 epochs**, `patience=3`).
4. **Seed Batch Splitting for Kaggle (0-4 & 5-9)**: Easily toggle `SEED_BATCH = 1` (seeds 0–4) or `SEED_BATCH = 2` (seeds 5–9) to avoid Kaggle execution timeouts.
5. **Multi-Tier Benchmark Evaluation**: Compute 3-tier metrics (Output Acc, Linear Probe Acc, NCC Acc) and class-mean geometry vs Oracle on $D_V$, $D_R$, $D_F$, and $D_T$.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    extract_features,
    compute_output_accuracy,
    compute_ncc_accuracy,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    print_metrics_summary_table,
    compute_class_mean_geometry,
    compare_class_mean_geometry,
    load_oracle_geometry,
    print_geometry_summary_table,
    MODEL_CONFIGS
)

from unlearning_methods import (
    unlearn_random_label,
    unlearn_neggrad_plus,
    unlearn_salun,
    unlearn_scrub,
    unlearn_tarun_unsir
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. CMF Head Reconstruction Utility

In [ ]:
# ============================================================
# CMF Head Reconstruction Utility & Early Stopping Engine
# ============================================================
from experiment_utils import reconstruct_cmf_head

# Early stopping configuration for Notebook 04a
MAX_EPOCHS_04A = 50  # Extended: higher ceiling for optimal convergence
PATIENCE_04A = 5     # Extended: longer patience to avoid premature stop on oscillations
MIN_EPOCHS_04A = 5    # Extended: ensure model has started learning

def find_artifact(filename, extra_dirs=None):
    """Dynamically search for an artifact across Kaggle input directories and local paths."""
    search_dirs = [
        "/kaggle/input/datasets/kiethe/cifar10-experimentnb1",
        "/kaggle/input/cifar10-experimentnb1",
        "/kaggle/input/datasets/kiethe/oracle-nb2-c10-experiment",
        "/kaggle/input/oracle-nb2-c10-experiment",
        "./artifacts/checkpoints",
        "./artifacts/splits",
        "./artifacts/metrics",
        "./artifacts",
        "/kaggle/working/artifacts/checkpoints",
        "/kaggle/working/artifacts/splits",
        "/kaggle/working/artifacts/metrics",
    ]
    if extra_dirs:
        search_dirs = list(extra_dirs) + search_dirs
    
    # Direct check in prioritized directories
    for d in search_dirs:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            return os.path.abspath(candidate)
            
    # Recursive walk in /kaggle/input and ./artifacts
    for root_dir in ["/kaggle/input", "./artifacts", "/kaggle/working"]:
        if os.path.exists(root_dir):
            for root, _, files in os.walk(root_dir):
                if filename in files:
                    return os.path.abspath(os.path.join(root, filename))
    return None


class EarlyStopException(Exception):
    """Exception raised to cleanly break out of unlearning loops upon early stopping trigger."""
    pass


def run_with_early_stop_04a(
    run_fn,
    model,
    cmf_loader,
    retain_loader,
    forget_loader,
    val_loader,
    num_classes,
    model_name="resnet18",
    seed=0,
    max_epochs=MAX_EPOCHS_04A,
    patience=PATIENCE_04A,
    min_epochs=MIN_EPOCHS_04A,
    device="cuda"
):
    """
    Runs unlearning algorithm with per-epoch CMF head re-tying (Algorithm 2 line 7)
    and Oracle proximity / validation-guided early stopping tracking.
    """
    # Load Oracle reference targets (geometry and metrics) if available
    oracle_ncc = None
    oracle_geom = None
    
    oracle_json_path = find_artifact(f"oracle_{model_name}_seed_{seed}.json")
    if oracle_json_path and os.path.exists(oracle_json_path):
        try:
            with open(oracle_json_path, "r") as f:
                oj = json.load(f)
            oracle_ncc = oj["test_metrics"]["ncc_forget"]
        except Exception:
            pass
            
    try:
        oracle_geom = load_oracle_geometry(model_name, seed)
    except Exception:
        oracle_geom = None

    state = {
        "best_dist": float("inf"),
        "best_state": None,
        "best_ncc": None,
        "best_cos": 1.0,
        "best_out_ret": 0.0,
        "no_improve": 0,
        "epoch": 0
    }

    def callback(m):
        # 1. Algorithm 2, line 7: Reconstruct CMF head from current encoder
        reconstruct_cmf_head(m, cmf_loader, num_classes=num_classes, device=device)
        
        state["epoch"] += 1
        m.eval()
        
        # 2. Extract features & compute representations
        r_feats, r_tgts = extract_features(m.encoder, retain_loader, device=device)
        f_feats, f_tgts = extract_features(m.encoder, forget_loader, device=device)
        ncc_forget = compute_ncc_accuracy(r_feats, r_tgts, f_feats, f_tgts, num_classes=num_classes)
        out_ret = compute_output_accuracy(m, retain_loader, device=device)
        
        retain_cos = 1.0
        if oracle_geom is not None:
            geom = compute_class_mean_geometry(m, retain_loader, forget_loader, num_classes=num_classes, device=device)
            cmp = compare_class_mean_geometry(geom, oracle_geom)
            retain_cos = cmp["retain"]["cos_mean"]
            dist = (1.0 - retain_cos) + (abs(ncc_forget - oracle_ncc) if oracle_ncc is not None else 0.0)
        elif oracle_ncc is not None:
            dist = abs(ncc_forget - oracle_ncc) + (1.0 - out_ret) * 0.5
        else:
            # Fallback: balance forget representation erasure and retain output utility
            dist = ncc_forget + (1.0 - out_ret) * 0.5

        if dist < state["best_dist"] - 1e-4:
            state["best_dist"] = dist
            state["best_ncc"] = ncc_forget
            state["best_cos"] = retain_cos
            state["best_out_ret"] = out_ret
            state["best_state"] = copy.deepcopy(m.state_dict())
            state["no_improve"] = 0
        else:
            state["no_improve"] += 1

        target_info = f" (target_oracle_ncc={oracle_ncc:.4f})" if oracle_ncc is not None else ""
        print(f"  [04a Early Stopping - Epoch {state['epoch']:02d}/{max_epochs:02d}] Retain Acc={out_ret:.4f} | NCC Forget={ncc_forget:.4f}{target_info} | Retain cos={retain_cos:.4f} | dist={dist:.4f} | no_improve={state['no_improve']}/{patience}")

        # Oracle-touch: stop immediately when |NCC_forget - oracle| <= 0.01
        if oracle_ncc is not None and abs(ncc_forget - oracle_ncc) <= 0.01:
            print(f"  [04a Oracle-touch] |NCC_forget({ncc_forget:.4f}) - oracle({oracle_ncc:.4f})| = {abs(ncc_forget - oracle_ncc):.4f} <= 0.01 at epoch {state['epoch']}. Stopping.")
            state["best_dist"] = dist
            state["best_ncc"] = ncc_forget
            state["best_cos"] = retain_cos
            state["best_out_ret"] = out_ret
            state["best_state"] = copy.deepcopy(m.state_dict())
            raise EarlyStopException(f"Oracle-touch stop at epoch {state['epoch']}")

        if state["epoch"] >= min_epochs and state["no_improve"] >= patience:
            print(f"  [04a Early Stopping] Triggered early stop at epoch {state['epoch']}/{max_epochs} (patience={patience}, min_epochs={min_epochs}).")
            raise EarlyStopException(f"Early stop triggered at epoch {state['epoch']}")

    try:
        unlearned = run_fn(model, callback)
    except EarlyStopException:
        unlearned = model

    if state["best_state"] is not None:
        unlearned.load_state_dict(state["best_state"])
        # Re-tie CMF head for restored best state
        reconstruct_cmf_head(unlearned, cmf_loader, num_classes=num_classes, device=device)
        best_ncc_str = f"{state['best_ncc']:.4f}" if state['best_ncc'] is not None else "N/A"
        print(f"  [04a Early Stopping] Restored best model (dist={state['best_dist']:.4f}, NCC Forget={best_ncc_str}, Retain cos={state['best_cos']:.4f}, Retain Acc={state['best_out_ret']:.4f}, total_epochs={state['epoch']}/{max_epochs})")
    else:
        reconstruct_cmf_head(unlearned, cmf_loader, num_classes=num_classes, device=device)

    return unlearned, state["epoch"]


## 2. Static CMF Unlearning Execution Loop Across All 5 Methods

> **Kaggle Execution Tip**: To avoid session timeouts with 30 epochs across 10 seeds and 5 unlearning methods, run in two parts:
> - **Part 1**: Set `SEED_BATCH = 1` for seeds `[0, 1, 2, 3, 4]`
> - **Part 2**: Set `SEED_BATCH = 2` for seeds `[5, 6, 7, 8, 9]`

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

# ============================================================
# SEED BATCH CONFIGURATION (Kaggle Session Optimization)
# Set SEED_BATCH = 1 for Part 1 (seeds 0-4)
# Set SEED_BATCH = 2 for Part 2 (seeds 5-9)
# Set SEED_BATCH = None for all seeds (0-9)
# ============================================================
SEED_BATCH = None  # Extended run: all seeds 0-9 in one go

if SEED_BATCH == 1:
    MODEL_SEEDS = [0, 1, 2, 3, 4]
    print(f"=== [Kaggle Run Part 1/2] Processing MODEL_SEEDS: {MODEL_SEEDS} ===")
elif SEED_BATCH == 2:
    MODEL_SEEDS = [5, 6, 7, 8, 9]
    print(f"=== [Kaggle Run Part 2/2] Processing MODEL_SEEDS: {MODEL_SEEDS} ===")
else:
    MODEL_SEEDS = cfg["model_seeds"]
    print(f"=== [Full Run] Processing all MODEL_SEEDS: {MODEL_SEEDS} ===")

METHODS = ["random_label", "neggrad_plus", "salun", "scrub", "tarun_unsir"]

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
from data_loader import load_raw_dataset as _load_raw_dataset
raw_train, raw_test = _load_raw_dataset(DATASET_NAME, root="./data")
if raw_test is None:
    raise ValueError(f"Dataset '{DATASET_NAME}' has no separate test split via load_raw_dataset; handle UTKFace via build_unlearning_splits instead.")

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/plots", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
split_filename = f"{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
candidate_splits = [
    f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{split_filename}",
    f"/kaggle/input/cifar10-experimentnb1/{split_filename}",
    f"/kaggle/input/datasets/kiethe/nb4a-c10-experiment/{split_filename}",
    f"/kaggle/input/nb4a-c10-experiment/{split_filename}",
    f"./artifacts/splits/{split_filename}"
]
split_file = None
for csp in candidate_splits:
    if os.path.exists(csp):
        split_file = os.path.abspath(csp)
        break
if split_file is None:
    split_file = find_artifact(split_filename)
    if split_file is None or not os.path.exists(split_file):
        split_file = f"./artifacts/splits/{split_filename}"
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact '{split_filename}' not found. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# DataLoaders
train_r_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
train_f_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

train_r_loader = DataLoader(train_r_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_f_loader = DataLoader(train_f_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

# CMF reconstruction dataset D (Algorithm 1/2): the unlearning training set D = D_R ∪ D_F,
# with eval transforms so class means are not affected by augmentation.
cmf_set = IndexedDataset(Subset(raw_train, split_info["train_indices"]), eval_tf)
cmf_loader = DataLoader(cmf_set, batch_size=BATCH_SIZE, shuffle=False)

all_04a_results = []

for seed in MODEL_SEEDS:
    orig_ckpt_fn = f"original_{MODEL_NAME}_seed_{seed}.pt"
    orig_metrics_fn = f"original_{MODEL_NAME}_seed_{seed}.json"
    
    orig_ckpt = find_artifact(orig_ckpt_fn)
    if orig_ckpt is None or not os.path.exists(orig_ckpt):
        local_orig_ckpt = f"./artifacts/checkpoints/{orig_ckpt_fn}"
        orig_ckpt = local_orig_ckpt if os.path.exists(local_orig_ckpt) else None
        
    orig_metrics_file = find_artifact(orig_metrics_fn)
    if orig_metrics_file is None or not os.path.exists(orig_metrics_file):
        local_orig_mf = f"./artifacts/metrics/{orig_metrics_fn}"
        orig_metrics_file = local_orig_mf if os.path.exists(local_orig_mf) else None
    
    if orig_ckpt is None or not os.path.exists(orig_ckpt):
        raise FileNotFoundError(f"Original checkpoint '{orig_ckpt_fn}' missing. Run Notebook 01 first.")
    print(f"Using original ckpt: {orig_ckpt} | metrics: {orig_metrics_file}")
        
    for method_name in METHODS:
        print(f"\n==================================================")
        print(f"--- Running 04a: CMF Static | Method: {method_name} | Model: {MODEL_NAME} | Seed: {seed} ---")
        print(f"==================================================")
        print(f"[Configuration]: Max Epochs = {MAX_EPOCHS_04A} | Patience = {PATIENCE_04A} | Min Epochs = {MIN_EPOCHS_04A}")
        
        # Load fresh original model
        model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(orig_ckpt, map_location=device))
        
        # 1. Algorithm 2, line 1: build the CMF head from D and freeze it
        reconstruct_cmf_head(model, cmf_loader, num_classes=NUM_CLASSES, device=device)
        
        start_time = time.time()
        
        # 2. Run unlearning algorithm on encoder with Early Stopping across max epochs
        print(f"[Step 2 - Running {method_name} with Early Stopping (Max Epochs: {MAX_EPOCHS_04A}, Patience: {PATIENCE_04A})]")
        if method_name == "random_label":
            def _run(m, cb):
                return unlearn_random_label(m, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, lr=0.013, epochs=MAX_EPOCHS_04A, device=device, epoch_end_callback=cb)
            unlearned_model, actual_epochs = run_with_early_stop_04a(_run, model, cmf_loader, retain_eval_loader, forget_eval_loader, val_loader, NUM_CLASSES, model_name=MODEL_NAME, seed=seed, max_epochs=MAX_EPOCHS_04A, patience=PATIENCE_04A, device=device)
        elif method_name == "neggrad_plus":
            def _run(m, cb):
                return unlearn_neggrad_plus(m, train_r_loader, train_f_loader, alpha=0.95, lr=0.01, epochs=MAX_EPOCHS_04A, device=device, epoch_end_callback=cb)
            unlearned_model, actual_epochs = run_with_early_stop_04a(_run, model, cmf_loader, retain_eval_loader, forget_eval_loader, val_loader, NUM_CLASSES, model_name=MODEL_NAME, seed=seed, max_epochs=MAX_EPOCHS_04A, patience=PATIENCE_04A, device=device)
        elif method_name == "salun":
            def _run(m, cb):
                return unlearn_salun(m, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, threshold=0.5, lr=0.013, epochs=MAX_EPOCHS_04A, device=device, epoch_end_callback=cb)
            unlearned_model, actual_epochs = run_with_early_stop_04a(_run, model, cmf_loader, retain_eval_loader, forget_eval_loader, val_loader, NUM_CLASSES, model_name=MODEL_NAME, seed=seed, max_epochs=MAX_EPOCHS_04A, patience=PATIENCE_04A, device=device)
        elif method_name == "scrub":
            def _run(m, cb):
                return unlearn_scrub(m, train_r_loader, train_f_loader, epochs=MAX_EPOCHS_04A, msteps=3, lr=0.006514, momentum=0.6490, weight_decay=0.003105, kd_T=3.888, alpha=0.9696, gamma=1.7154, batch_size=32, forget_batch_size=128, device=device, epoch_end_callback=cb)
            unlearned_model, actual_epochs = run_with_early_stop_04a(_run, model, cmf_loader, retain_eval_loader, forget_eval_loader, val_loader, NUM_CLASSES, model_name=MODEL_NAME, seed=seed, max_epochs=MAX_EPOCHS_04A, patience=PATIENCE_04A, device=device)
        elif method_name == "tarun_unsir":
            cmf_callback = lambda m: reconstruct_cmf_head(m, cmf_loader, num_classes=NUM_CLASSES, device=device)
            unlearned_model = unlearn_tarun_unsir(model, train_r_loader, forget_loader=train_f_loader, num_classes=NUM_CLASSES, noise_lr=0.9345, noise_epochs=2, noise_steps=15, impair_lr=6.638e-5, impair_epochs=2, impair_batches=14, repair_lr=9.582e-4, repair_epochs=1, device=device, epoch_end_callback=cmf_callback)
            reconstruct_cmf_head(unlearned_model, cmf_loader, num_classes=NUM_CLASSES, device=device)
            actual_epochs = "fixed"
            
        runtime = time.time() - start_time
        
        # Save unlearned CMF checkpoint
        save_ckpt = f"./artifacts/checkpoints/cmf_04a_ext_{method_name}_{MODEL_NAME}_seed_{seed}.pt"
        torch.save(unlearned_model.state_dict(), save_ckpt)
        
        # Evaluate 3-tier metrics
        val_metrics = {"output_val": compute_output_accuracy(unlearned_model, val_loader, device=device)}
        test_metrics = evaluate_three_tier_metrics(
            model=unlearned_model, probe_train_loader=retain_eval_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device, test_eval_loader=test_loader, nonmember_loader=val_loader
        )
        
        # Class-mean geometry: global_mean -> class_mean directions, compared with the Oracle
        geom = compute_class_mean_geometry(
            unlearned_model, retain_eval_loader, forget_eval_loader, num_classes=NUM_CLASSES, device=device
        )
        geometry_vs_oracle = compare_class_mean_geometry(geom, load_oracle_geometry(MODEL_NAME, seed))

        res = {
            "variant": "04a_cmf_static",
            "method": method_name,
            "model_name": MODEL_NAME,
            "seed": seed,
            "split_seed": SPLIT_SEED,
            "runtime_sec": runtime,
            "max_epochs": MAX_EPOCHS_04A,
            "trained_epochs": actual_epochs,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics,
            "geometry_vs_oracle": geometry_vs_oracle
        }
        all_04a_results.append(res)
        
        res_path = f"./artifacts/metrics/cmf_04a_ext_{method_name}_{MODEL_NAME}_seed_{seed}.json"
        with open(res_path, "w") as f:
            json.dump(res, f, indent=2)
            
        print(f"[{method_name}] Trained Epochs: {actual_epochs} (Max: {MAX_EPOCHS_04A}) | Runtime: {runtime:.2f}s")
        print(f"[{method_name}] Geometry vs Oracle | Retain cos: {geometry_vs_oracle['retain']['cos_mean']:.4f} "
              f"| Retain norm ratio: {geometry_vs_oracle['retain']['norm_ratio_mean']:.4f} "
              f"| Forget cos: {geometry_vs_oracle['forget']['cos_mean']:.4f} "
              f"| Forget norm ratio: {geometry_vs_oracle['forget']['norm_ratio_mean']:.4f}")
        print(f"[{method_name}] 04a Retain (D_R) Output Acc: {test_metrics['output_retain']:.4f} | Forget (D_F) Output Acc: {test_metrics['output_forget']:.4f} | Test (D_T) Acc: {test_metrics.get('output_test', 0.0):.4f}")
        print(f"[{method_name}] 04a Retain (D_R) LP Acc:     {test_metrics['lp_retain']:.4f} | Forget (D_F) LP Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
        print(f"[{method_name}] 04a Retain (D_R) NCC Acc:    {test_metrics['ncc_retain']:.4f} | Forget (D_F) NCC Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
        print(f"[{method_name}] Saved 04a checkpoint & metrics to {res_path}")
        
        # Feature space & decision boundary visualization for first seed
        if seed == MODEL_SEEDS[0]:
            vis_save = f"./artifacts/plots/features_04a_cmf_{method_name}_{MODEL_NAME}_seed_{seed}.png"
            visualize_feature_space_and_boundaries(
                model=unlearned_model,
                retain_loader=retain_eval_loader,
                forget_loader=forget_eval_loader,
                num_classes=NUM_CLASSES,
                title=f"Feature Space: 04a CMF Static ({method_name}, {MODEL_NAME}, Seed {seed})",
                device=device,
                save_path=vis_save
            )

# ============================================================
# Final Consolidated Benchmark Metrics Summary Table
# ============================================================
batch_suffix = f"_batch{SEED_BATCH}" if SEED_BATCH is not None else ""
summary_df = print_metrics_summary_table(
    results=all_04a_results,
    title=f"04a CMF Static Representation Unlearning Metrics Summary (Seeds {MODEL_SEEDS[0]}-{MODEL_SEEDS[-1]})",
    save_csv_path=f"./artifacts/metrics/summary_04a_cmf_{MODEL_NAME}{batch_suffix}.csv"
)

# ============================================================
# Class-Mean Direction Geometry vs Oracle (per method)
# ============================================================
geometry_df = print_geometry_summary_table(
    results=all_04a_results,
    title=f"04a CMF Static: Class-Mean Direction Geometry vs Oracle (D_R) (Seeds {MODEL_SEEDS[0]}-{MODEL_SEEDS[-1]})",
    save_csv_path=f"./artifacts/metrics/summary_geometry_04a_cmf_{MODEL_NAME}{batch_suffix}.csv"
)